# 01. 과거 구성종목을 포함한 S&P 500 10년 OHLCV 수집
현재 구성종목뿐 아니라 분석 기간 중 한 번이라도 S&P 500에 포함됐던 종목을 수집합니다. API를 호출하는 유일한 노트북이므로 최초 수집 또는 명시적인 재수집 때만 실행합니다.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from src.collect_prices import END_DATE, START_DATE, collect_sp500_index, make_df
from src.get_tickers import get_historical_sp500_universe, get_sp500_universe
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f'수집 구간: {START_DATE} ~ {END_DATE} (끝 미포함)')

In [ ]:
# 현재 명단은 회사명/섹터 메타데이터용, 과거 명단은 생존자 편향 없는 수집 대상용입니다.
current_universe = get_sp500_universe()
historical_universe = get_historical_sp500_universe(START_DATE, END_DATE)

sp500_universe = historical_universe.merge(
    current_universe[['ticker', 'company', 'sector']],
    on='ticker',
    how='left',
)
sp500_universe['company'] = sp500_universe['company'].fillna(sp500_universe['ticker'])
sp500_universe['sector'] = sp500_universe['sector'].fillna('Unknown')
sp500_universe = (
    sp500_universe.drop_duplicates('ticker')
    .sort_values('ticker')
    .reset_index(drop=True)
)

snp500_tickers = sp500_universe['ticker'].tolist()
current_tickers = set(current_universe['ticker'])
historical_tickers = set(snp500_tickers)
print(f'현재 구성종목: {len(current_tickers)}종목')
print(f'과거 편출 포함 수집 대상: {len(historical_tickers)}종목')
print(f'현재 명단에 없던 과거 종목: {len(historical_tickers - current_tickers)}종목')
display(sp500_universe.head())

In [ ]:
# 오래 걸리는 셀입니다. yfinance -> Yahoo chart -> Tiingo 순으로 복구합니다.
final_df_raw, final_missing_df = make_df(snp500_tickers, start=START_DATE, end=END_DATE, universe=sp500_universe)
print(final_df_raw.shape, final_missing_df.shape)

In [ ]:
# Beta 계산에만 사용할 S&P 500 지수(^GSPC)를 종목 패널과 별도로 수집합니다.
sp500_beta_df = collect_sp500_index(start=START_DATE, end=END_DATE)
print(sp500_beta_df.shape)
display(sp500_beta_df.head())

In [ ]:
final_df_raw.to_parquet(RAW_DIR / 'final_df_raw.parquet', index=False)
sp500_beta_df.to_parquet(RAW_DIR / 'sp500_beta_df.parquet', index=False)
final_missing_df.to_csv(RAW_DIR / 'final_missing_df.csv', index=False)
sp500_universe.to_csv(RAW_DIR / 'sp500_universe.csv', index=False)
print(f'raw 저장 완료: {RAW_DIR}')

In [ ]:
final_df_raw.head()

In [ ]:
# 실제 데이터 형태
display(final_df_raw.head(10))

# 전체 데이터 규모
print("데이터 크기:", final_df_raw.shape)
print("종목 수:", final_df_raw["Ticker"].nunique())
print("날짜 범위:", final_df_raw["Date"].min(), "~", final_df_raw["Date"].max())

# 수집하지 못한 종목과 사유
display(final_missing_df)

In [ ]:
print(
    "Close 0 이하:",
    (final_df_raw["Close"] <= 0).sum(),
)

In [ ]:
import pandas as pd 
display(
    pd.DataFrame({
        "결측값": final_df_raw[
            ["Open", "High", "Low", "Volume"]
        ].isna().sum(),
        "0인 값": final_df_raw[
            ["Open", "High", "Low", "Volume"]
        ].eq(0).sum(),
    })
)

# 안정성의 베타지표 계산하기 위해 필요한 benchmark, snp500 지수
- 벤치마크가 없으면 종목 자체의 흔들림인 변동성은 계산할 수 있으나, 시장 대비 민감도인 베타 계산을 위해 필요
- Beta = 1: 시장과 비슷한 정도로 움직임
- Beta > 1: 시장보다 더 민감하게 움직임
- 0 < Beta < 1: 시장보다 덜 민감하게 움직임
- Beta < 0: 시장과 반대 방향으로 움직이는 경향

In [ ]:
display(sp500_beta_df.head())
print(sp500_beta_df.shape)